In [ ]:
import os, sys, time, pickle, glob
from pathlib import Path

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IS_KAGGLE:
    print('Kaggle GPU')
    os.environ['TF_USE_LEGACY_KERAS'] = '1'
    os.environ['CNN_EPOCHS'] = '1' # Set 1 untuk Dry Run
    !pip install -q tf-keras
    
    SRC_PATH = '/kaggle/input/cnn-src/src'
    if not os.path.exists(SRC_PATH):
        SRC_PATH = '/kaggle/input/datasets/kefaskurniajonathan/cnn-src-2/src'
    
    sys.path.insert(0, SRC_PATH)
    print(f'Added {SRC_PATH} to sys.path')
    
    os.environ['CNN_DATA_DIR'] = '/kaggle/input/datasets/kefaskurniajonathan/data-intel-cnn'
    
    REPO_ROOT = Path('/kaggle/working')
    os.chdir(REPO_ROOT)
else:
    def _find_root(marker='requirements.txt'):
        p = Path(os.getcwd())
        while p != p.parent:
            if (p / marker).exists(): return p
            p = p.parent
        return p
    REPO_ROOT = _find_root()
    os.chdir(REPO_ROOT)
    sys.path.insert(0, str(REPO_ROOT / 'src'))
    print('Running Locally. Root:', REPO_ROOT)

In [ ]:
import tensorflow as tf
from tensorflow import keras

from cnn.train_keras import (
    build_conv2d_model, build_locally_connected_model,
    get_data_loaders, train, IMG_SIZE
)

tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
MODELS_DIR  = "models/cnn"
HISTORY_DIR = "models/cnn/history"

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

In [ ]:
print("Loading data...")
train_ds, val_ds = get_data_loaders(img_size=IMG_SIZE)
train_ds_lc, val_ds_lc = get_data_loaders(img_size=(64, 64))

In [ ]:
FILTER_CONFIGS = {
    "base":  {2: [32, 64],       3: [32, 64, 128]},
    "large": {2: [64, 128],      3: [64, 128, 256]},
}

VARIATIONS = []
for n_layers in [2, 3]:
    for f_type in ["base", "large"]:
        for k_size in [3, 5]:
            for p_type in ["max", "avg"]:
                name = f"cnn-{n_layers}L-{f_type}-k{k_size}-{p_type}"
                VARIATIONS.append({
                    "name": name,
                    "num_conv_layers": n_layers,
                    "filters": FILTER_CONFIGS[f_type][n_layers],
                    "kernel_sizes": [k_size] * n_layers,
                    "pooling": p_type,
                })
print(f"Total arsitektur: {len(VARIATIONS)}")

In [ ]:
total_variations = len(VARIATIONS)
histories = {}
for i, v in enumerate(VARIATIONS):
    name = v['name']
    # --- PROGRESS LOG ---
    print(f'\n' + '='*50)
    print(f'[ MODEL {i+1} dari {total_variations+1} ]')
    print(f'Melatih: {name}')
    print('='*50 + '\n')
    
    kwargs = {k: val for k, val in v.items() if k != 'name'}
    model = build_conv2d_model(**kwargs)
    history = train(model, train_ds, val_ds, model_name=name)
    
    histories[name] = history.history
    with open(os.path.join(HISTORY_DIR, f'{name}_history.pkl'), 'wb') as f:
        pickle.dump(history.history, f)

with open(os.path.join(MODELS_DIR, 'all_conv2d_histories.pkl'), 'wb') as f:
    pickle.dump(histories, f)
print("\nOK Semua Conv2D selesai.")

In [ ]:
print("\nTraining LocallyConnected2D...")
lc_model = build_locally_connected_model()
lc_history = train(lc_model, train_ds_lc, val_ds_lc, model_name="lc-model")
with open(os.path.join(HISTORY_DIR, "lc-model_history.pkl"), "wb") as f:
    pickle.dump(lc_history.history, f)
print("\nOK Training Selesai!")